In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/Airline_review.csv")

In [ ]:
df = df.dropna(subset=["Overall_Rating"])

In [ ]:
df.shape

In [ ]:
df["Recommended"].unique()

In [ ]:
df["full_review_text"] = df["Review_Title"] + " " + df["Review"]
df[["Review_Title", "Review", "full_review_text"]].head()

In [ ]:
import re
def clean_text(text):
    
    text = str(text)
    
    # lowercase
    text = text.lower()
    
    # remove urls
    text = re.sub(r"http\S+", "", text)
    
    # remove special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [ ]:
df["clean_review_text"] = df["full_review_text"].apply(clean_text)

In [ ]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

In [ ]:
def remove_stopwords(text):

    words = text.split()

    filtered_words = [word for word in words if word not in stop_words]

    return " ".join(filtered_words)

In [ ]:
df["processed_text"] = df["clean_review_text"].apply(remove_stopwords)
df[["clean_review_text","processed_text"]].head()


In [ ]:
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):

    words = text.split()

    lemmatized = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(lemmatized)

In [ ]:
df["final_text"] = df["processed_text"].apply(lemmatize_text)

In [ ]:
df[["processed_text", "final_text"]].head()

In [ ]:
df.to_csv("../data/clean_airline_reviews.csv", index=False)

In [ ]:
df["Overall_Rating"] = pd.to_numeric(df["Overall_Rating"], errors="coerce")

In [ ]:
def severity_label(rating):

    if rating <= 2:
        return "Critical"
    
    elif rating <= 4:
        return "High"
    
    elif rating <= 6:
        return "Medium"
    
    else:
        return "Low"

In [ ]:
df["severity_label"] = df["Overall_Rating"].apply(severity_label)

In [ ]:
df["severity_label"].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["severity_id"] = label_encoder.fit_transform(df["severity_label"])

In [ ]:
df[["severity_label","severity_id"]].head()